# Q9: Data Cleaning
**Topic:** Coding-question | **Difficulty:** Easy | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
Implement a comprehensive data cleaning pipeline in Python.

In [ ]:
import numpy as np
import pandas as pd
from typing import Optional


def create_dirty_dataset() -> pd.DataFrame:
    """Create a sample dataset with common data quality issues."""
    np.random.seed(42)
    n = 100
    data = {
        'age': np.random.randint(18, 80, n).astype(float),
        'salary': np.random.normal(50000, 15000, n),
        'department': np.random.choice(['Engineering', 'Sales', 'HR', 'engineering', ' Sales ', None], n),
        'experience': np.random.randint(0, 40, n).astype(float),
        'rating': np.random.uniform(1, 5, n),
    }
    df = pd.DataFrame(data)
    # Inject issues
    df.loc[np.random.choice(n, 10, replace=False), 'age'] = np.nan
    df.loc[np.random.choice(n, 5, replace=False), 'salary'] = -5000  # invalid
    df.loc[np.random.choice(n, 3, replace=False), 'age'] = 200  # outlier
    df.loc[np.random.choice(n, 8, replace=False), 'salary'] = np.nan
    # Duplicate rows
    df = pd.concat([df, df.iloc[:5]], ignore_index=True)
    return df


df = create_dirty_dataset()
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")
df.head(10)

In [ ]:
class DataCleaner:
    """A reusable data cleaning pipeline."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.log: list[str] = []
    
    def remove_duplicates(self) -> 'DataCleaner':
        """Remove duplicate rows."""
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = before - len(self.df)
        self.log.append(f"Removed {removed} duplicate rows")
        return self
    
    def standardize_strings(self, columns: list[str]) -> 'DataCleaner':
        """Strip whitespace and lowercase string columns."""
        for col in columns:
            if self.df[col].dtype == 'object':
                self.df[col] = self.df[col].str.strip().str.lower()
        self.log.append(f"Standardized strings in {columns}")
        return self
    
    def handle_missing(self, strategy: dict[str, str]) -> 'DataCleaner':
        """Handle missing values per column.
        
        strategy: dict mapping column name to 'mean', 'median', 'mode', or 'drop'.
        """
        for col, method in strategy.items():
            if method == 'mean':
                self.df[col].fillna(self.df[col].mean(), inplace=True)
            elif method == 'median':
                self.df[col].fillna(self.df[col].median(), inplace=True)
            elif method == 'mode':
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
            elif method == 'drop':
                self.df = self.df.dropna(subset=[col])
        self.log.append(f"Handled missing values: {strategy}")
        return self
    
    def clip_outliers(self, column: str, lower: float, upper: float) -> 'DataCleaner':
        """Clip values outside [lower, upper] range."""
        outliers = ((self.df[column] < lower) | (self.df[column] > upper)).sum()
        self.df[column] = self.df[column].clip(lower, upper)
        self.log.append(f"Clipped {outliers} outliers in {column} to [{lower}, {upper}]")
        return self
    
    def remove_invalid(self, column: str, condition: str) -> 'DataCleaner':
        """Remove rows where a condition is True (e.g., 'salary < 0')."""
        before = len(self.df)
        self.df = self.df.query(f"not ({condition})")
        removed = before - len(self.df)
        self.log.append(f"Removed {removed} invalid rows where {condition}")
        return self
    
    def summary(self) -> pd.DataFrame:
        """Print cleaning log and return cleaned DataFrame."""
        print("=== Cleaning Log ===")
        for entry in self.log:
            print(f"  ✓ {entry}")
        print(f"\nFinal shape: {self.df.shape}")
        print(f"Remaining missing: {self.df.isnull().sum().sum()}")
        return self.df

In [ ]:
# --- Run the pipeline ---
cleaner = DataCleaner(df)
clean_df = (
    cleaner
    .remove_duplicates()
    .standardize_strings(['department'])
    .remove_invalid('salary', 'salary < 0')
    .clip_outliers('age', lower=18, upper=100)
    .handle_missing({
        'age': 'median',
        'salary': 'mean',
        'department': 'mode'
    })
    .summary()
)

clean_df.head()

## Key Takeaways
- Always **inspect data first** (`.info()`, `.describe()`, `.isnull().sum()`)
- Build a **reusable pipeline** with method chaining — it's readable and reproducible
- Handle issues in this order: duplicates → type fixes → invalid values → outliers → missing values
- Document every cleaning step for reproducibility